# Manam volcano ? a landscape in three dimensions

Explore the active volcanic island of Manam in Papua New Guinea using real elevation data. Drag to orbit, scroll to zoom, switch to a north-up overhead view, or reveal 100 m contours. All three axes use kilometres, with **no vertical exaggeration**; colours encode elevation.

The 1,807 m summit is an independent reference. The gridded model is not a survey and does not have the same resolution or accuracy as a local volcanic hazard map.

## Data and reproducibility

This notebook uses a small crop of the 1 arc-second SRTM HGT tile `S05E145`, distributed by [Mapzen Elevation Tiles on AWS](https://elevation-tiles-prod.s3.amazonaws.com/skadi/S05/S05E145.hgt.gz). SRTM is a digital elevation model created from NASA/NGA radar observations and distributed by the [U.S. Geological Survey](https://www.usgs.gov/centers/eros/science/usgs-eros-archive-digital-elevation-shuttle-radar-topography-mission-srtm). It is not bathymetry or a live volcano-monitoring product. The tile is bundled as `data/manam_dem.npz`, so ordinary runs are offline; set `REFRESH_DATA = True` to fetch and rebuild the source crop. The source tile SHA-256 and access date are checked and recorded. The displayed crop covers 8 ? 8 km and is bilinearly sampled to 281 ? 281 points.

The HTML export is generated from this notebook and written to `output/topology_3d.html`. Plot, browser controls, and page layout are kept here as the source of truth.


In [1]:
from pathlib import Path
import gzip
import hashlib
import json
import math
import sys
import urllib.request
import numpy as np
import plotly.graph_objects as go

current = Path.cwd().resolve()
project_root = next((p for p in (current, *current.parents)
                     if (p / 'site_export.py').is_file() and (p / 'notebooks').is_dir()), None)
if project_root is None:
    raise RuntimeError('Run this notebook from the repository or notebooks directory.')
sys.path.insert(0, str(project_root)) if str(project_root) not in sys.path else None
from site_export import write_plot_page

SUMMIT_LAT, SUMMIT_LON = -4.08, 145.037
SUMMIT_M = 1807
RADIUS_KM, GRID_SIZE = 4.0, 281
REFRESH_DATA = False
SOURCE_URL = 'https://elevation-tiles-prod.s3.amazonaws.com/skadi/S05/S05E145.hgt.gz'
SOURCE_SHA256 = 'd9e1c626e769a6e1ecc5dad0ad133cb12995bf19c968410e0cc325730914cd58'
data_file = project_root / 'data/manam_dem.npz'
cache_dir = project_root / '.cache/terrain'
km_lat = 111.195
km_lon = km_lat * math.cos(math.radians(SUMMIT_LAT))
bounds = (SUMMIT_LAT - RADIUS_KM / km_lat, SUMMIT_LAT + RADIUS_KM / km_lat,
          SUMMIT_LON - RADIUS_KM / km_lon, SUMMIT_LON + RADIUS_KM / km_lon)


In [2]:
# Rebuild only on request; the committed crop makes normal runs fully offline.
if REFRESH_DATA:
    cache_dir.mkdir(parents=True, exist_ok=True)
    tile_file = cache_dir / 'S05E145.hgt.gz'
    if not tile_file.exists():
        request = urllib.request.Request(SOURCE_URL, headers={'User-Agent': 'phey.app-terrain/1.0'})
        with urllib.request.urlopen(request, timeout=120) as response:
            raw_download = response.read()
        if hashlib.sha256(raw_download).hexdigest() != SOURCE_SHA256:
            raise ValueError('Downloaded SRTM tile checksum does not match the recorded source.')
        temporary = tile_file.with_suffix('.part')
        temporary.write_bytes(raw_download)
        temporary.replace(tile_file)
    compressed = tile_file.read_bytes()
    if hashlib.sha256(compressed).hexdigest() != SOURCE_SHA256:
        raise ValueError('Cached SRTM tile checksum mismatch; remove the corrupt cache and retry.')
    raw = gzip.decompress(compressed)
    tile = np.frombuffer(raw, dtype='>i2')
    if tile.size != 3601 * 3601:
        raise ValueError(f'Unexpected SRTM HGT sample count: {tile.size}')
    tile = tile.reshape((3601, 3601)).astype(np.int16)
    tile_lat = -4 - np.arange(3601, dtype=np.float64) / 3600
    tile_lon = 145 + np.arange(3601, dtype=np.float64) / 3600
    rows = np.flatnonzero((tile_lat >= bounds[0]) & (tile_lat <= bounds[1]))
    cols = np.flatnonzero((tile_lon >= bounds[2]) & (tile_lon <= bounds[3]))
    if not rows.size or not cols.size:
        raise ValueError('The selected SRTM tile does not cover the Manam crop.')
    crop = tile[rows[0]:rows[-1]+1, cols[0]:cols[-1]+1]
    source_lat = tile_lat[rows[0]:rows[-1]+1]
    source_lon = tile_lon[cols[0]:cols[-1]+1]
    if np.any(crop == -32768):
        raise ValueError('The Manam crop contains SRTM void samples; refusing to invent elevations.')
    provenance = json.dumps({
        'dataset': 'SRTM 1 arc-second elevation data',
        'distribution': 'Mapzen Elevation Tiles (Skadi), public AWS object',
        'tile': 'S05E145', 'url': SOURCE_URL, 'accessed': '2026-10-27',
        'source_sha256': SOURCE_SHA256,
        'format': 'SRTM HGT, 3601 x 3601 signed 16-bit samples; WGS84 grid, elevations in metres',
        'attribution': 'SRTM data courtesy of the U.S. Geological Survey; data originated with NASA/NGA.'})
    np.savez_compressed(data_file, elevation_m=crop, latitude=source_lat,
                        longitude=source_lon, provenance=provenance)

if not data_file.is_file():
    raise FileNotFoundError(f'Missing bundled terrain crop: {data_file}. Set REFRESH_DATA = True to build it.')
with np.load(data_file, allow_pickle=False) as data:
    crop = data['elevation_m'].astype(np.float64)
    source_lat, source_lon = data['latitude'], data['longitude']
    provenance = json.loads(str(data['provenance']))
if provenance.get('source_sha256') != SOURCE_SHA256 or provenance.get('tile') != 'S05E145':
    raise ValueError('Unexpected cached dataset; rebuild with REFRESH_DATA = True')
if crop.shape != (len(source_lat), len(source_lon)):
    raise ValueError('Terrain grid and coordinate arrays do not align.')
if np.any(crop == -32768) or not np.isfinite(crop).all():
    raise ValueError('Terrain contains void or invalid elevations.')
if not (np.all(np.diff(source_lat) < 0) and np.all(np.diff(source_lon) > 0)):
    raise ValueError('Unexpected SRTM coordinate order.')
print(f'Loaded source crop: {crop.shape}; {data_file.stat().st_size / 1024:.1f} KiB')


Loaded source crop: (261, 262); 86.5 KiB


In [3]:
# Bilinear sampling uses the original one-arc-second grid without altering the source.
lat = np.linspace(bounds[1], bounds[0], GRID_SIZE)
lon = np.linspace(bounds[2], bounds[3], GRID_SIZE)
if not (source_lat[-1] <= lat[-1] < lat[0] <= source_lat[0]
        and source_lon[0] <= lon[0] < lon[-1] <= source_lon[-1]):
    raise ValueError('The bundled crop does not cover this extent. Refresh the data for the chosen bounds.')

def sample_elevation(latitudes, longitudes):
    rows = (source_lat[0] - np.asarray(latitudes)) / (source_lat[0] - source_lat[1])
    cols = (np.asarray(longitudes) - source_lon[0]) / (source_lon[1] - source_lon[0])
    i = np.clip(np.floor(rows).astype(int), 0, crop.shape[0] - 2)
    j = np.clip(np.floor(cols).astype(int), 0, crop.shape[1] - 2)
    fy, fx = rows - i, cols - j
    return ((1-fy) * ((1-fx)*crop[i,j] + fx*crop[i,j+1])
            + fy * ((1-fx)*crop[i+1,j] + fx*crop[i+1,j+1]))

latitude, longitude = np.meshgrid(lat, lon, indexing='ij')
height_m = np.round(sample_elevation(latitude, longitude), 1)
x_km = np.round((lon - SUMMIT_LON) * km_lon, 5)
y_km = np.round((lat - SUMMIT_LAT) * km_lat, 5)
height_km = height_m / 1000
summit_dem = float(sample_elevation(SUMMIT_LAT, SUMMIT_LON))
if height_m.shape != (GRID_SIZE, GRID_SIZE) or not np.isfinite(height_m).all():
    raise ValueError('The display grid is incomplete or invalid.')
if not 1000 < summit_dem < 1900:
    raise ValueError(f'Unexpected elevation at Manam summit coordinate: {summit_dem:.0f} m')
print(f'Displayed terrain range: {height_m.min():,.0f} to {height_m.max():,.0f} m')
print(f'DEM at summit coordinate: {summit_dem:,.0f} m; independent summit reference: {SUMMIT_M:,} m')


Displayed terrain range: -204 to 1,770 m
DEM at summit coordinate: 1,670 m; independent summit reference: 1,807 m


In [4]:
# Horizontal and vertical axes are all in km; aspectmode=data avoids exaggeration.
BG, FG, MUTED = '#0d1117', '#eaf0f5', '#a4b2c1'
colors = [[0, '#173f52'], [.15, '#347564'], [.35, '#78a36f'],
          [.55, '#b7b17c'], [.75, '#b89168'], [.90, '#d7c6a9'], [1, '#faf6e9']]
home_camera = dict(eye=dict(x=1.3, y=1.65, z=.85), up=dict(x=0, y=0, z=1),
                   center=dict(x=0, y=0, z=-.12), projection=dict(type='perspective'))
top_camera = dict(eye=dict(x=0, y=0, z=2.5), up=dict(x=0, y=1, z=0),
                  center=dict(x=0, y=0, z=0), projection=dict(type='orthographic'))
fig = go.Figure(go.Surface(
    x=x_km.astype(np.float32), y=y_km.astype(np.float32), z=height_km.astype(np.float32),
    surfacecolor=height_m.astype(np.float32), colorscale=colors, cmin=0, cmax=SUMMIT_M,
    customdata=np.stack((np.abs(np.round(latitude, 5)), np.round(longitude, 5)), axis=-1).astype(np.float32),
    hovertemplate='Elevation %{z:.3f} km<br>%{customdata[0]:.5f}? S ? %{customdata[1]:.5f}? E<extra>Terrain</extra>',
    colorbar=dict(title=dict(text='Elevation ? m', side='top'), orientation='h',
                  x=.5, xanchor='center', y=-.08, len=.60, thickness=10,
                  tickvals=[0, 500, 1000, 1500, 1800], outlinewidth=0),
    contours=dict(z=dict(show=False, start=0, end=1.8, size=.1,
                         usecolormap=False, color='#45515c', width=1, highlight=False)),
    lighting=dict(ambient=.55, diffuse=.85, roughness=.9, specular=.15, fresnel=.15),
    lightposition=dict(x=10000, y=15000, z=20000), name='Terrain'))
fig.add_trace(go.Scatter3d(x=[0], y=[0], z=[SUMMIT_M/1000], mode='markers+text',
    marker=dict(size=4, color='#f3d37b', line=dict(width=1, color=BG)),
    text=[f'Manam ? {SUMMIT_M:,} m'], textposition='top center', textfont=dict(size=12, color=FG),
    hovertemplate=f'Summit reference: {SUMMIT_M:,} m<br>DEM here: {summit_dem:,.0f} m<extra>Manam</extra>',
    showlegend=False))
axis = dict(showbackground=False, gridcolor='#28323d', zeroline=False,
            tickfont=dict(size=10, color=MUTED), showspikes=False)
fig.update_layout(template='plotly_dark', paper_bgcolor=BG, plot_bgcolor=BG,
    font=dict(family='Arial, sans-serif', color=FG, size=12), height=700,
    margin=dict(l=10, r=10, t=60, b=80), showlegend=False,
    scene=dict(bgcolor=BG, aspectmode='data', camera=home_camera,
        xaxis=dict(**axis, title='East ? km', range=[-4,4], dtick=2),
        yaxis=dict(**axis, title='North ? km', range=[-4,4], dtick=2),
        zaxis=dict(**axis, title='Altitude ? km', range=[-.25,2.0], dtick=.5),
    ),
    updatemenus=[dict(type='buttons', direction='right', x=0, xanchor='left', y=1.11,
        yanchor='top', bgcolor='#1b2733', bordercolor='#455569', font=dict(size=12),
        showactive=False, buttons=[
            dict(label='Reset 3D', method='relayout', args=[{'scene.camera':home_camera}]),
            dict(label='Overhead', method='relayout', args=[{'scene.camera':top_camera}]),
            dict(label='Contours', method='restyle', args=[{'contours.z.show':True}, [0]],
                 args2=[{'contours.z.show':False}, [0]])])])
config = dict(responsive=True, scrollZoom=True, displaylogo=False, displayModeBar=False)
fig.show(config=config)


In [5]:
# The notebook defines the page and browser interactions; export uses shared legal navigation.
output_file = project_root / 'output/topology_3d.html'
plot_html = fig.to_html(include_plotlyjs=False, full_html=False, div_id='manam-terrain',
    config=config, default_height='100%', post_script="""
const terrain = document.getElementById('manam-terrain');
const desktopCamera = JSON.parse(JSON.stringify(terrain.layout.scene.camera));
const mobileCamera = JSON.parse(JSON.stringify(desktopCamera));
for (const axis of ['x', 'y', 'z']) mobileCamera.eye[axis] *= 1.5;
let wasCompact;
function keyboardControls() {
  terrain.querySelectorAll('.updatemenu-button').forEach(button => {
    button.setAttribute('role', 'button');
    button.setAttribute('tabindex', '0');
    button.setAttribute('aria-label', button.textContent);
    if (button.textContent === 'Contours') {
      button.setAttribute('aria-pressed', String(!!terrain.data[0].contours.z.show));
    }
    button.onkeydown = event => {
      if (event.key === 'Enter' || event.key === ' ') {
        event.preventDefault();
        button.dispatchEvent(new MouseEvent('click', {bubbles: true}));
      }
    };
  });
}
function fitTerrain() {
  const compact = window.innerWidth < 600;
  const home = compact ? mobileCamera : desktopCamera;
  const update = {height: compact ? 530 : Math.max(580, Math.min(800, window.innerHeight - 240)),
    'scene.xaxis.title.text': compact ? 'E ? km' : 'East ? km',
    'scene.yaxis.title.text': compact ? 'N ? km' : 'North ? km',
    'updatemenus[0].buttons[0].args': [{'scene.camera': home}]};
  if (compact !== wasCompact) update['scene.camera'] = home;
  wasCompact = compact;
  Plotly.relayout(terrain, update).then(keyboardControls);
}
terrain.on('plotly_afterplot', keyboardControls);
fitTerrain();
window.addEventListener('resize', fitTerrain);
""")
html = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Manam Volcano 3D Topography | phey.app</title>
  <meta name="description" content="Explore Manam volcano in Papua New Guinea in three dimensions with real SRTM elevation data, a north-up overhead view and 100-metre contours.">
  <style>
    * {{ box-sizing: border-box; }}
    html, body {{ margin: 0; background: {BG}; color: {FG}; font-family: system-ui, sans-serif; }}
    main {{ max-width: 1600px; margin: auto; padding: 24px 32px 18px; }}
    a {{ color: #b8d2dd; text-underline-offset: 3px; }}
    .back {{ font-size: 13px; text-decoration: none; color: {MUTED}; }}
    .heading {{ display: flex; justify-content: space-between; align-items: end; gap: 20px; margin: 28px 0 16px; }}
    .eyebrow {{ color: #cbb48c; font-size: 11px; letter-spacing: .2em; text-transform: uppercase; margin: 0 0 8px; }}
    h1 {{ font-size: clamp(32px, 4vw, 52px); line-height: 1.05; margin: 0; font-weight: 500; letter-spacing: -.04em; }}
    .subtitle {{ color: {MUTED}; margin: 10px 0 0; font-size: 14px; }}
    .stats {{ display: flex; gap: 32px; flex-shrink: 0; }}
    .stats span {{ display: block; color: {MUTED}; font-size: 11px; margin-top: 6px; }}
    .stats strong {{ font-size: 20px; font-weight: 500; }}
    .plot {{ border-top: 1px solid #28323d; border-bottom: 1px solid #28323d; padding-top: 12px; }}
    .hint, .source {{ color: {MUTED}; font-size: 12px; line-height: 1.65; }}
    .hint {{ margin: 14px 0 0; }}
    details {{ margin-top: 16px; max-width: 100ch; }}
    summary {{ cursor: pointer; color: #cbd6df; }}
    .updatemenu-button:focus-visible rect {{ stroke: #e9c58d; stroke-width: 2px; }}
    a:focus-visible, summary:focus-visible {{ outline: 2px solid #e9c58d; outline-offset: 4px; }}
    @media(max-width: 600px) {{
      main {{ padding: 18px 12px; }}
      .heading {{ display: block; margin-top: 24px; }}
      .stats {{ gap: 24px; margin-top: 22px; }}
      .stats strong {{ font-size: 17px; }}
    }}
  </style>
</head>
<body>
<main>
  <a class="back" href="../index.html">? Back to phey.app</a>
  <header class="heading">
    <div><p class="eyebrow">Bismarck Sea / Papua New Guinea</p>
      <h1>Manam</h1><p class="subtitle">An active volcanic island, in three dimensions.</p></div>
    <div class="stats" aria-label="Terrain facts">
      <div><strong>1,807 m</strong><span>Summit reference</span></div>
      <div><strong>8 ? 8 km</strong><span>Area explored</span></div>
      <div><strong>1 : 1</strong><span>True height scale</span></div>
    </div>
  </header>
  <div class="plot" role="region" aria-label="Interactive Manam volcano elevation model">{plot_html}</div>
  <p class="hint">Drag to orbit ? Scroll or pinch to zoom ? Overhead faces north ? Contours every 100 m.<br>
    Colours show elevation, not land cover. The summit pin is a reference elevation; the grid at its coordinate reads {summit_dem:,.0f} m.</p>
  <details class="source"><summary>About the terrain &amp; data sources</summary>
    <p>An 8 ? 8 km crop of 1 arc-second <a href="https://www.usgs.gov/centers/eros/science/usgs-eros-archive-digital-elevation-shuttle-radar-topography-mission-srtm">SRTM elevation data</a>,
    using the S05E145 HGT tile distributed through <a href="https://github.com/tilezen/joerd">Mapzen Elevation Tiles</a>.
    The grid is a digital elevation model, not bathymetry or a surveyed hazard map. Horizontal distances use a local latitude-adjusted projection;
    all axes use kilometres without vertical exaggeration. The displayed terrain ranges from {height_m.min():,.0f} to {height_m.max():,.0f} m.
    The summit marker ({SUMMIT_M:,} m) is a separate reference; the grid reads {summit_dem:,.0f} m there.</p>
    <p>SRTM data courtesy of the U.S. Geological Survey; the elevation data originated with NASA/NGA.
    <a href="https://github.com/tilezen/joerd/blob/master/docs/data-sources.md">Data sources</a> ?
    <a href="https://github.com/regnumproequo/p00/blob/main/notebooks/topology_3d.ipynb">Notebook &amp; method</a>.
    Source tile accessed 27 October 2026.</p>
  </details>
</main>
</body>
</html>"""
html = '
'.join(line.rstrip() for line in html.splitlines()) + '
'
write_plot_page(output_file, html)
print(f'Exported {output_file.name} ({output_file.stat().st_size / 1024**2:.2f} MiB)')


SyntaxError: unterminated string literal (detected at line 105) (743476409.py, line 105)